# Plot meshes

In [1]:
from matplotlib import colormaps as cm
import numpy as np
import pyvista as pv

from mammos_mumag.mesh import Mesh, find_mesh
from mammos_mumag.simulation import Simulation

In [2]:
import meshio

In [3]:
Mesh("cube80_3plat_grains12_gsize40")._write_from_keeper("plot_mesh.med")

In [4]:
mesh = meshio.read("plot_mesh.med")

In [5]:
def convert_to_multiblock(mesh: meshio.Mesh) -> pv.MultiBlock:
    subregion_names = mesh.cell_tags
    subregion_pv = dict()
    points = mesh.points
    connectivity = mesh.cells[1].data
    mesh.cell_data_to_sets("cell_tags")
    for key, val in mesh.cell_sets_dict.items():
        subregion_tag = int(key.removeprefix("set-cell_tags-"))
        if subregion_tag not in subregion_names:
            continue
        subregion_name = subregion_names[subregion_tag][0]
        subregion_cells = mesh.cell_sets_dict[f"set-cell_tags-{subregion_tag}"]["tetra"]
        new_cell_block = meshio.CellBlock(cell_type="tetra", data=connectivity[subregion_cells])
        subregion_pv[subregion_name] = pv.from_meshio(meshio.Mesh(points=points, cells=[new_cell_block]))
    return pv.MultiBlock(subregion_pv)

In [6]:
mltbk = convert_to_multiblock(mesh)
mltbk

MultiBlock (0x7f333224d360)
  N Blocks:   15
  X Bounds:   -1.296e+02, 1.294e+02
  Y Bounds:   -1.289e+02, 1.290e+02
  Z Bounds:   -1.296e+02, 1.296e+02

In [7]:
pl = pv.Plotter()
for i in range(15):
    if i in [12, 13, 14]: # filter unwanted regions [grain boundary, shell-inner, shell-outer]
        continue
    subregion = str(i+1)
    pl.add_mesh(mltbk[subregion], style='wireframe', color='k', line_width=1)
pl.show()

Widget(value='<iframe src="http://localhost:43027/index.html?ui=P_0x7f3330db7a90_0&reconnect=auto" class="pyvi…

## Tags:
- `-20` shell
- `-19` air
- `-18` grain boundary
- `-17` ... `-6` grains
- `-5` ... `-1` nothing
- `0` lines and triangles (all surfaces)